In [0]:
import pandas as pd

In [0]:
billings_df = pd.read_csv("../../data/raw/billings.csv")
billings_df.head()


## Drop **Duplicates**

In [0]:
billings_df = billings_df.loc[:, ~billings_df.columns.duplicated()]
billings_df.drop_duplicates(inplace=True)

## Fixing Datatypes and standardising

In [0]:
billings_df.info()

### Fixing Date Columns

In [0]:
date_cols = [
    'Renewal_Month', 'Last_Years_Date_Paid', 'Proforma_Date',
    'Registration_Date', 'Prospect_Renewal_Date', 'Closed_Date',
    'Last_Renewal', 'DateTime_Out'
]
for col in date_cols:
    billings_df[col] = pd.to_datetime(billings_df[col], dayfirst=True, errors='coerce')

### Convert Discount_Amount to Float

In [0]:
# billings_df['Discount_Amount'] = pd.to_numeric(
#     billings_df['Discount_Amount'].astype(str).str.replace('%', '', regex=False).replace('nan', pd.NA),
#     errors='coerce'
# )
billings_df['Discount_Amount'] = (
    billings_df['Discount_Amount']
    .astype(str)
    .str.replace('%', '', regex=False)
)

billings_df['Discount_Amount'] = pd.to_numeric(
    billings_df['Discount_Amount'],
    errors='coerce'
) / 100

### Convert Columns to Categorical

In [0]:
categorical_cols = [
    'Proforma_Account_Stage',     # 7 unique
    'Proforma_Audit_Status',      # 31 unique
    'Proforma_Membership_Status', # 4 unique
    'Band',                       # 14 unique
    'Last_Band',                  # 14 unique
    'Prospect_Status',            # 46 unique
    'Prospect_Outcome',           # 3 unique
    'Payment_Method',             # 5 unique
    'Connection_Group',           # 6 unique
    'Tenure_Group',               # 4 unique
    'Anchor_Group',               # 6 unique
]

for col in categorical_cols:
    print(col, '\n')
    print(billings_df[col].value_counts())

In [0]:
for col in categorical_cols:
    billings_df[col] = billings_df[col].astype('category')


### Convert Boolean-like Columns

In [0]:
billings_df['Current_Auto_Renewal_Flag'].value_counts()

In [0]:
billings_df['Current_Auto_Renewal_Flag'] = (
    billings_df['Current_Auto_Renewal_Flag'].map({'y': True, 'n': False}).astype('boolean')
)
billings_df['Current_World_Pay_Token'] = (
    billings_df['Current_World_Pay_Token'].map({'y': True, 'n': False}).astype('boolean')
)

billings_df['Proforma_Auto_Renewal'] = billings_df['Proforma_Auto_Renewal'].astype('boolean')
billings_df['Proforma_World_Pay_Token'] = billings_df['Proforma_World_Pay_Token'].astype('boolean')

print("Converted 4 columns to boolean dtype")

In [0]:
billings_df['Current_World_Pay_Token'].value_counts()

### Enforce Integer Types with Nullable Int

In [0]:
nullable_int_candidates = [
    'Connection_Qty', 'Starting_Connection_Qty',
    'Current_Anchorings', 'Proforma_Approved_Lists'
]
for col in nullable_int_candidates:
    if billings_df[col].dropna().apply(lambda x: x == int(x)).all():
        billings_df[col] = billings_df[col].astype('Int64')




### Final Dtypes Verification

In [0]:

dtype_summary = pd.DataFrame({
    'Column': billings_df.columns,
    'Dtype': billings_df.dtypes.astype(str).values,
    'Non_Null': billings_df.notna().sum().values,
    'Null_Count': billings_df.isna().sum().values,
    'Nunique': [billings_df[c].nunique() for c in billings_df.columns]
})
display(dtype_summary)

## Null Handling

In [0]:
null_summary = pd.DataFrame({
    'null_count': billings_df.isnull().sum(),
    'null_percentage': (billings_df.isnull().sum() / len(billings_df)) * 100
})
null_summary

In [0]:
# 1. Drop 100% null column
billings_df.drop(columns=['Last_Years_Date_Paid'], inplace=True)
print("Dropped Last_Years_Date_Paid (100% null)")

### Fill numeric nulls with 0

In [0]:
# 2. Fill structurally meaningful numeric nulls with 0
fill_zero_cols = [
    'Discount_Amount',           # 88.9% null — no discount applied
    'Connection_Net',            # 93.4% null — no add-on connections
    'Connection_Qty',            # 93.4% null — no add-on connections
    'Starting_Connection_Net',   # 93.0% null — no starting connections
    'Starting_Connection_Qty',   # 93.0% null — no starting connections
    # 'Total_Net_Paid',            # 17.1% null — Churned/Open, nothing paid
    'Last_Total_Net_Paid',       # 39.6% null — no prior renewal
    'Last_Connections',          # 39.6% null — no prior renewal
    'Last_Years_Price',          #  7.4% null — no prior year price
]
for col in fill_zero_cols:
    billings_df[col] = billings_df[col].fillna(0)


In [0]:
#Total_Net_Paid is same as Total_Amount
billings_df['Total_Net_Paid'] = billings_df['Total_Amount']

### Fill boolean nulls with False 

In [0]:

billings_df['Proforma_Auto_Renewal'] = billings_df['Proforma_Auto_Renewal'].fillna(False)
billings_df['Proforma_World_Pay_Token'] = billings_df['Proforma_World_Pay_Token'].fillna(False)

billings_df['Current_Anchor_List'] = billings_df['Current_Anchor_List'].fillna('None')


### Fill categorical nulls with "Unknown"

In [0]:
fill_unknown_cols = [
    'Band',                        #  24 nulls
    'Connection_Group',            # 126 nulls
    'Anchor_Group',                # 126 nulls
    'Proforma_Membership_Status',  # 126 nulls
    'Tenure_Group',                # 1,018 nulls
    'Proforma_Account_Stage',      # 9,229 nulls
    'Proforma_Audit_Status',       # 9,229 nulls
    'Last_Band',                   # 48,311 nulls — no prior renewal
]
for col in fill_unknown_cols:
    if billings_df[col].dtype.name == 'category':
        billings_df[col] = billings_df[col].cat.add_categories('Unknown')
    billings_df[col] = billings_df[col].fillna('Unknown')


### Fill low-null numeric columns with mean

In [0]:
fill_mean_cols = [
    'Renewal_Score_At_Release',  # 126 nulls
    '#_of_Connection',           # 126 nulls
    'Proforma_Approved_Lists',   # 126 nulls
    'Tenure_Years',              # 1,018 nulls
]

for col in fill_mean_cols:
    mean_val = round(billings_df[col].mean())
    fill_values = billings_df.groupby('Co_Ref')[col].transform('mean')
    if billings_df[col].dtype == 'Int64':
        fill_values = fill_values.round().astype('Int64')
    billings_df[col] = billings_df[col].fillna(fill_values)

In [0]:
null_summary = pd.DataFrame({
    'null_count': billings_df.isnull().sum(),
    'null_percentage': (billings_df.isnull().sum() / len(billings_df)) * 100
})
null_summary

In [0]:
billings_df.to_csv("../../data/processed/billings.csv", index=False)